```markdown
# 🌟 Demystifying Classification in Deep Learning

Up to this point, our models have learned **numerical relationships** (like predicting a house price based on its size). But what happens if we feed our model an image and ask: **"Is this a dog, a cat, or a bird?"**

This is a **Classification Problem**.

Let's break down how neural networks make these decisions using a simple, human-readable pipeline!

---

## 🗺️ The Classification Pipeline

Think of the classification pipeline as a factory assembly line:

```
[ Raw Input ]  ──►  [ Neural Network ]  ──►  [ Logits ]  ──►  [ Softmax / Predictions ]
 (e.g. Image)        (Learns Patterns)       (Raw Scores)        (Final Probabilities)
```

1. **Input:** The raw features (like pixels of an image or coordinates of a point).
2. **Neural Network:** The brain. It uses weights and activation functions to extract features.
3. **Logits:** The final raw, unfiltered output scores from the network.
4. **Loss Function (Cross-Entropy):** The supervisor that measures how far off our predictions are from reality.
5. **Optimizer (Backpropagation):** The mechanic that fine-tunes the network so it makes better guesses next time.
```

So far our model learns numerical relationships but what if we give an image to our model and ask if it is a dog or a cat?

That's a classification problem.

## Understanding the classification pipeline:

1. Input (Image or other features)
2. Neural Networks (Learns patterns using weight and activation)
3. Logits (Raw scores for each class)
4. Loss Function (Measures how well the predictions match the labels)
5. Backpropagation + Optimizer (Calculates gradients and updates parameters)

The training process is familiar. The main new concepts are the logits, class labels, and classification loss functions.

```markdown
### ⚖️ Logits vs. Probabilities

Why does the network output **logits** instead of percentages (probabilities) directly?

* **Logits** can be any real number: positive, negative, or zero (e.g., `2.5`, `-1.0`).
* **Probabilities** must always add up to exactly `1.0` (or `100%`) and cannot be negative.

Keeping them as raw logits is **mathematically cleaner and more stable** for the computer during training. When we want to make a final prediction, we simply look for the largest logit!
```

**What are Logits?**

Suppose our network classifies an image into three classes:
- cat
- dog
- bird

The final layer might produce:

logits = torch.tensor([[2.5, 0.8, -1.0]])


THese are raw scores not probabilities.

Class	Logit
Cat	2.5
Dog	0.8
Bird	-1.0

The largest score is for Cat, so the predicted class is Cat.

e.g: predicted_class = logits.argmax(dim=1)
     print(predicted_class)

and output will be tensor[0]

Here, class index 0 represents Cat.
Why not output probabilities directly? Neural networks can produce raw scores, and the appropriate loss function can transform them internally when needed. This is often more numerically stable.

In [1]:
# we use nn.CrossEntropyLoss() as a loss function in classification commonly

import torch
import torch.nn as nn

logits = torch.tensor([[2.5, 0.8, -1.0]])

target = torch.tensor([0])

criterion = nn.CrossEntropyLoss()
loss = criterion(logits, target)

print("Loss: ", loss.item())

Loss:  0.19299839437007904


What does cross-entropy do?
It penalizes the model when it assigns insufficient probability to the correct class. A confident, correct prediction generally has low loss; a confident, incorrect prediction generally has high loss.
Important: Don't apply Softmax before CrossEntropyLoss. PyTorch combines the relevant operations internally for numerical stability.

# Let's build a complete classification network

In [4]:
import torch
import torch.nn as nn

# three clusters of two-dim data
X = torch.tensor([
    [1.0, 1.0],
    [1.2, 0.8],
    [0.8, 1.2],
    [-1.0, -1.0],
    [-1.2, -0.8],
    [-0.8, -1.2],
    [1.0, -1.0],
    [1.2, -0.8],
    [0.8, -1.2]
])

# Class labels: 0, 1, or 2
y = torch.tensor([0, 0, 0, 1, 1, 1, 2, 2, 2])

# two input features, 8 hidden neurons, three output classes
model = nn.Sequential(
    nn.Linear(2, 8),
    nn.ReLU(),
    nn.Linear(8, 3)
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

for epoch in range(300):
  # Forward pass
  logits = model(X)

  # Calculate classification loss
  loss = criterion(logits, y)

  # Backpropagation and parameter update
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  if epoch % 30 == 0:
    print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f}")

# Evaluate predictions
with torch.no_grad():
  logits = model(X)
  predictions = logits.argmax(dim=1)
  accuracy = (predictions == y).float().mean()

print("Predictions:", predictions.tolist())
print("Actual labels:", y.tolist())
print("Training accuracy:", accuracy.item())

Epoch   0 | Loss: 1.0977
Epoch  30 | Loss: 0.7297
Epoch  60 | Loss: 0.3942
Epoch  90 | Loss: 0.1990
Epoch 120 | Loss: 0.1138
Epoch 150 | Loss: 0.0741
Epoch 180 | Loss: 0.0529
Epoch 210 | Loss: 0.0402
Epoch 240 | Loss: 0.0320
Epoch 270 | Loss: 0.0263
Predictions: [0, 0, 0, 1, 1, 1, 2, 2, 2]
Actual labels: [0, 0, 0, 1, 1, 1, 2, 2, 2]
Training accuracy: 1.0
